In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
!pip install -q segmentation_models_pytorch
# TO DO
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import numpy as np
import os
import matplotlib.pyplot as plt
from PIL import Image
from segmentation_models_pytorch import Unet

device = "cuda" if torch.cuda.is_available() else "cpu"
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

target_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])


def remap_mask(mask):
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

class SUIMDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.target_transform = target_transform
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))])
        self.mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.png')])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        mask = remap_mask(mask)

        return image, mask

path = "/kaggle/input/q3-stage3-2026/dataset"
image_dir = os.path.join(path, "images")
mask_dir = os.path.join(path, "masks")

dataset = SUIMDataset(image_dir, mask_dir, transform=transform, target_transform=target_transform)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

batch_images, batch_masks = next(iter(train_loader))
print(f"Batch images shape: {batch_images.shape}")
print(f"Batch masks shape: {batch_masks.shape}")

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for i in range(4):
    axes[0, i].imshow(batch_images[i].cpu().numpy().transpose(1, 2, 0))
    axes[0, i].set_title(f"Image {i}")
    axes[0, i].axis('off')

    axes[1, i].imshow(batch_masks[i].cpu().numpy().squeeze(), cmap='tab10')
    axes[1, i].set_title(f"Mask {i}")
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()
#ive been on this code for a whole hour

In [ ]:
# TO DO
class UNetModel(nn.Module):
    def __init__(self, num_classes=8):
        super(UNetModel, self).__init__()
        self.model = Unet(
            encoder_name="efficientnet-b1",
            encoder_weights="imagenet",
            in_channels=3,
            classes=num_classes
        )

    def forward(self, x):
        return self.model(x)

model = UNetModel(num_classes=8).to(device)
print(f"Model moved to {device}")

In [ ]:
# TO DO
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device).squeeze(1).long()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device).squeeze(1).long()
            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

    return running_loss / len(dataloader)

In [ ]:
# TO DO
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 3
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, train_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()
#this didnt finish and my credits are finished so i dont even get to see the answer just like q1 i dont know what to do how do i check my work if i cant see it im will to pay now but if i open another page they might say im cheating

In [ ]:
# TO DO
def visualize_predictions(model, dataloader, device, num_samples=3):
    model.eval()

    with torch.no_grad():
        for i, (images, masks) in enumerate(dataloader):
            if i >= num_samples:
                break

            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            images = images.cpu().numpy()
            masks = masks.cpu().numpy()
            predictions = predictions.cpu().numpy()

            fig, axes = plt.subplots(1, 3, figsize=(15, 5))

            for j in range(3):
                axes[j].imshow(np.transpose(images[j], (1, 2, 0)))
                axes[j].set_title(['Image', 'Ground Truth', 'Prediction'][j])
                axes[j].axis('off')

            cmap = plt.cm.get_cmap('tab10', 8)
            axes.imshow(masks, cmap=cmap, vmin=0, vmax=7)
            axes.imshow(predictions, cmap=cmap, vmin=0, vmax=7)

            plt.tight_layout()
            plt.show()

visualize_predictions(model, train_loader, device)